# Reprodução — Transformers e LLMs no Google Colab (gratuito)

Notebook da reprodução de *From COCOMO to GPT* (Alaswad et al., IEEE Access, 2026).

**Antes de começar:** `Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4`.

O Colab gratuito pode desconectar. Por isso tudo é salvo no seu Google Drive: se cair, rode as células de novo, e os scripts continuam da última época / última issue.

## 1. Google Drive e repositório

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = "https://github.com/SEU_USUARIO/Story-Points-Estimation.git"   # <- troque pelo seu fork
BRANCH = "reproducao"                                                    # branch onde está a pasta repro/
DRIVE_DIR = "/content/drive/MyDrive/SEE-LLM-repro"          # onde ficam checkpoints e resultados

In [ ]:
import os
if not os.path.exists("/content/SEE-LLM"):
    !git clone -q -b $BRANCH $REPO_URL /content/SEE-LLM
%cd /content/SEE-LLM
!git pull -q
!pip install -q -r repro/requirements-transformers.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Dados

Os splits e os textos perturbados já estão versionados no repositório. Aqui baixamos os datasets (versões fixadas por commit e conferidas por hash) e geramos os arquivos processados.

In [ ]:
!python repro/01_download_data.py
!python repro/02_prepare_data.py

## 3. Resultados persistentes no Drive

A pasta `repro/results` vira um link para o Drive, então nada se perde se a sessão cair.

In [ ]:
import shutil, pathlib
res = pathlib.Path("repro/results"); drive_res = pathlib.Path(DRIVE_DIR) / "results"
drive_res.mkdir(parents=True, exist_ok=True)
if res.exists() and not res.is_symlink():
    shutil.copytree(res, drive_res, dirs_exist_ok=True); shutil.rmtree(res)
if not res.exists():
    res.symlink_to(drive_res)
print("resultados em", drive_res)

## 4. Transformers (BERT, RoBERTa, GPT-2, GPT2SP)

Na T4, cada modelo leva aproximadamente de 30 min a 2 h por dataset, dependendo de quando o early stopping para. Rode um modelo por vez se preferir. Modelos que já têm predições são pulados.

In [ ]:
!python repro/20_run_transformers.py --dataset agile --models BERT RoBERTa GPT-2 GPT2SP --ckpt-dir "$DRIVE_DIR/checkpoints"

In [ ]:
!python repro/20_run_transformers.py --dataset neo --models BERT RoBERTa GPT-2 GPT2SP --ckpt-dir "$DRIVE_DIR/checkpoints"

In [ ]:
# Generalização (RQ3) — opcional, cada protocolo re-treina os modelos
!python repro/20_run_transformers.py --protocol cross --ckpt-dir "$DRIVE_DIR/checkpoints"

## 4b. Variante fiel ao CÓDIGO original (só Agile)

Verifica se os números de Transformers da Tabela 14 saem do código publicado
(split aleatório 80/20, 20 épocas fixas, sem validação). Para o ML, a mesma
verificação (`11_run_ml_original_code.py`) já reproduziu KNN e Regressão Linear
exatamente, e roda na CPU.

In [ ]:
!python repro/21_run_transformers_original_code.py --models BERT RoBERTa GPT-2 --ckpt-dir "$DRIVE_DIR/checkpoints"

## 5. (Opcional) LLMs abertos via Ollama na GPU do Colab

A T4 roda modelos de ~8B parâmetros bem mais rápido que um notebook sem GPU. O script de LLM é retomável: as respostas vão para `results/llm_raw/` (no Drive).

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!ollama pull llama3.1:8b

In [ ]:
!python repro/30_run_llm.py --provider ollama --model llama3.1:8b --dataset agile neo

## 5b. Qualidade da entrada com o Laya (juiz aberto, mesma API do Jev)

Na GPU T4 o Laya julga cada issue em dezenas de milissegundos. Na CPU de um notebook, leva cerca de 1 s ou mais por issue.

In [ ]:
!pip install -q "laya[serve]"
import subprocess, os, time
env = dict(os.environ, LAYA_MODELS="typed-decisions", LAYA_DEVICE="cuda", LAYA_PRELOAD="1")
subprocess.Popen(["laya-serve"], env=env, stdout=open("/content/laya.log", "w"), stderr=subprocess.STDOUT)
time.sleep(90)   # baixa e carrega o checkpoint
!tail -n 5 /content/laya.log

In [ ]:
# teste rápido; se funcionar, rode a célula seguinte
!python repro/50_quality_judge.py --engine laya --dataset agile --limit 5

In [ ]:
!python repro/50_quality_judge.py --engine laya                 # 2.188 issues de avaliação
# !python repro/50_quality_judge.py --engine laya --subset all  # dataset inteiro (~44 mil issues)
!python repro/51_quality_report.py report

## 6. Métricas

In [ ]:
!python repro/90_evaluate.py